## 1. Data Preprocessing

In [1]:
import sys
import os
import dotenv
import numpy as np
import pandas as pd

dotenv.load_dotenv("../credentials.env")
from travel_insurance_claim.data_ingestion import DataIngestion

In [2]:
obj = DataIngestion(mongo_uri=os.environ['MONGODB_URI'], database_name=os.environ['DB_NAME'], collection_name=os.environ['COLLECTION_NAME'])
data = obj.read_data()

# 2. Feature Engineering  

In [3]:
data.head()

,Agency,Agency Type,Distribution Channel,Product Name,Claim,Duration,Destination,Net Sales,Commision (in value),Gender,Age
0,CBH,Travel Agency,Offline,Comprehensive Plan,No,186,MALAYSIA,-29.0,9.57,F,81
1,CBH,Travel Agency,Offline,Comprehensive Plan,No,186,MALAYSIA,-29.0,9.57,F,71
2,CWT,Travel Agency,Online,Rental Vehicle Excess Insurance,No,65,AUSTRALIA,-49.5,29.70,NaN,32
3,CWT,Travel Agency,Online,Rental Vehicle Excess Insurance,No,60,AUSTRALIA,-39.6,23.76,NaN,32
4,CWT,Travel Agency,Online,Rental Vehicle Excess Insurance,No,79,ITALY,-19.8,11.88,NaN,41


In [4]:
data.Agency.value_counts()

Agency
EPX    35119
CWT     8580
C2B     8267
JZI     6329
SSI     1056
JWT      749
RAB      725
LWC      689
TST      528
KML      392
ART      331
CCR      194
CBH      101
TTW       98
CSR       86
ADM       82
Name: count, dtype: int64

In [5]:
data['Agency Type'].value_counts()

Agency Type
Travel Agency    45869
Airlines         17457
Name: count, dtype: int64

In [6]:
data['Distribution Channel'].value_counts()

Distribution Channel
Online     62219
Offline     1107
Name: count, dtype: int64

In [7]:
data['Product Name'].value_counts()

Product Name
Cancellation Plan                       18630
2 way Comprehensive Plan                13158
Rental Vehicle Excess Insurance          8580
Basic Plan                               5469
Bronze Plan                              4049
1 way Comprehensive Plan                 3331
Value Plan                               2715
Silver Plan                              2249
Annual Silver Plan                       1423
Ticket Protector                         1056
Travel Cruise Protect                     527
Comprehensive Plan                        364
Gold Plan                                 352
24 Protect                                247
Single Trip Travel Protect Gold           204
Annual Gold Plan                          194
Premier Plan                              194
Single Trip Travel Protect Silver         173
Annual Travel Protect Gold                100
Annual Travel Protect Silver               86
Individual Comprehensive Plan              74
Single Trip Travel Pr

In [8]:
print(data['Destination'].value_counts()); print(data['Destination'].nunique())

Destination
SINGAPORE                 13255
MALAYSIA                   5930
THAILAND                   5894
CHINA                      4796
AUSTRALIA                  3694
                          ...  
SEYCHELLES                    1
SAMOA                         1
NIGERIA                       1
DOMINICAN REPUBLIC            1
BOSNIA AND HERZEGOVINA        1
Name: count, Length: 149, dtype: int64
149


# 3. Handling Missing Values

In [9]:
data.isna().sum()

Agency                      0
Agency Type                 0
Distribution Channel        0
Product Name                0
Claim                       0
Duration                    0
Destination                 0
Net Sales                   0
Commision (in value)        0
Gender                  45107
Age                         0
dtype: int64

In [10]:
data = data.drop("Gender", axis=1)

Dropping gender since random genders cannot be imputed.

# 4. Dropping Unwanted Columns
`Gender` was already dropped above (about 70% missing and cannot be imputed sensibly). `Agency` is dropped as well, since `Agency Type` already carries the useful, lower-cardinality information.

In [11]:
data = data.drop("Agency", axis=1)
data.columns.tolist()

['Agency Type',
 'Distribution Channel',
 'Product Name',
 'Claim',
 'Duration',
 'Destination',
 'Net Sales',
 'Commision (in value)',
 'Age']

# 5. Encoding Categorical Columns
- `Claim` (target) and the two binary columns are mapped to 0/1.
- `Product Name` and `Destination` have many categories (26 and 149), so categories below 1% of rows are grouped into `Other` before one-hot encoding. This avoids hundreds of sparse columns and does not use the target, so there is no leakage.

In [12]:
# Target: Yes/No -> 1/0
data["Claim"] = data["Claim"].map({"No": 0, "Yes": 1})

# Binary categorical columns
data["Agency Type"] = data["Agency Type"].map({"Airlines": 0, "Travel Agency": 1})
data["Distribution Channel"] = data["Distribution Channel"].map({"Offline": 0, "Online": 1})

data[["Claim", "Agency Type", "Distribution Channel"]].isna().sum()

Claim                   0
Agency Type             0
Distribution Channel    0
dtype: int64

In [13]:
def group_rare(series, min_freq=0.01):
    """Replace categories that make up less than min_freq of rows with 'Other'."""
    freq = series.value_counts(normalize=True)
    keep = freq[freq >= min_freq].index
    return series.where(series.isin(keep), "Other")

data["Product Name"] = group_rare(data["Product Name"], 0.01)
data["Destination"] = group_rare(data["Destination"], 0.01)

print(data["Product Name"].nunique(), data["Destination"].nunique())

11 18


In [14]:
# One-hot encode the remaining nominal columns
data = pd.get_dummies(data, columns=["Product Name", "Destination"], drop_first=True, dtype=int)

# 6. Final Check

In [15]:
print(data.shape)
print(data.select_dtypes(include="object").columns.tolist())  # should be empty
print(data.isna().sum().sum())                                 # should be 0
data.head()

(63326, 34)
[]
0


,Agency Type,Distribution Channel,Claim,Duration,Net Sales,Commision (in value),Age,Product Name_2 way Comprehensive Plan,Product Name_Annual Silver Plan,Product Name_Basic Plan,...,Destination_MALAYSIA,Destination_MYANMAR,Destination_Other,Destination_PHILIPPINES,Destination_SINGAPORE,"Destination_TAIWAN, PROVINCE OF CHINA",Destination_THAILAND,Destination_UNITED KINGDOM,Destination_UNITED STATES,Destination_VIET NAM
0,1,0,0,186,-29.0,9.57,81,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1,1,0,0,186,-29.0,9.57,71,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,1,1,0,65,-49.5,29.70,32,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,1,0,60,-39.6,23.76,32,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,1,0,79,-19.8,11.88,41,0,0,0,...,0,0,1,0,0,0,0,0,0,0


## Summary of preprocessing
1. Loaded the data from MongoDB and explored the categorical columns.
2. Dropped `Gender` (too many missing values) and `Agency` (redundant with `Agency Type`).
3. Encoded `Claim`, `Agency Type` and `Distribution Channel` as 0/1.
4. Grouped rare `Product Name` and `Destination` categories (<1% of rows) into `Other`, then one-hot encoded both columns.
5. No train/test split was done; all columns are now numeric and ready for the next step.